# Fase inicial: Extracción de métricas de fichero de audio.

In [2]:
import pandas as pd
import os

# Cargar las etiquetas del dataset ASVspoof
ruta_protocolo = '../data/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt'
columnas = ['speaker_id', 'file_name', 'system_id', 'attack_id', 'label']
df_etiquetas = pd.read_csv(ruta_protocolo, sep=' ', names=columnas)


# Filtrar solo los nombres de archivo y si son reales o falsos
df_etiquetas = df_etiquetas[['file_name', 'label']]

print(df_etiquetas.head())

      file_name     label
0  LA_T_1138215  bonafide
1  LA_T_1271820  bonafide
2  LA_T_1272637  bonafide
3  LA_T_1276960  bonafide
4  LA_T_1341447  bonafide


## Prueba 1

Esta es basada en el artículo de Mael Fabien - https://maelfabien.github.io/machinelearning/Speech9/#6-mel-frequency-cepstral-coefficients-mfcc

El artículo menciona 14 puntos, pero dejé fuera algunas cosas por eficiencia computacional y relevancia:

* Tempogram (Punto 9) y Polyfeatures (Punto 8): Son características orientadas casi exclusivamente al Análisis de Información Musical (MIR), para detectar acordes o ritmos musicales. En detección de voz falsa aportan mucho "ruido" computacional y poco valor.

* Frecuencia Fundamental o Pitch (Punto 11): Es valiosísima para detectar emociones o distinguir género, pero extraerla con precisión en Python requiere algoritmos iterativos (como YIN o pYIN) que son computacionalmente costosos. Ralentizaría drásticamente tu procesamiento de miles de audios en esta primera prueba.

* Jitter (Punto 12): Requiere tener primero una extracción perfecta de la Frecuencia Fundamental, por lo que entra en la misma categoría de coste computacional.

In [3]:
import librosa
import numpy as np
import pandas as pd
import os

def extract_features(file_path):
    # Intentar cargar el archivo de audio con manejo de errores
    try:
        # sr=None preserva la tasa de muestreo original del archivo
        y, sr = librosa.load(file_path, sr=None)
    except Exception as e:
        print(f"Error al procesar {file_path}: {e}")
        return None

    features = {}
    
    # 1. Estadisticas basicas de la forma de onda
    features['signal_mean'] = np.mean(y)
    features['signal_std'] = np.std(y)
    
    # 2. Energia (Root Mean Square Energy)
    rmse = librosa.feature.rms(y=y)
    features['rmse_mean'] = np.mean(rmse)
    
    # 3. Zero-Crossing Rate (ZCR)
    zcr = librosa.feature.zero_crossing_rate(y)
    features['zcr_mean'] = np.mean(zcr)
    
    # 4. Tempo (BPM)
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    # Dependiendo de la version de librosa, tempo puede ser un float o un array
    features['tempo_bpm'] = tempo[0] if isinstance(tempo, np.ndarray) else tempo
    
    # 5. MFCCs (13 coeficientes es el estandar mas comun para voz)
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    # Colapsamos la matriz temporal calculando la media y desviacion de cada coeficiente
    for i in range(1, 14):
        features[f'mfcc_{i}_mean'] = np.mean(mfccs[i-1])
        features[f'mfcc_{i}_std'] = np.std(mfccs[i-1])
        
    # 6. Caracteristicas Espectrales
    spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    features['spectral_centroid_mean'] = np.mean(spectral_centroid)
    
    spectral_bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)
    features['spectral_bandwidth_mean'] = np.mean(spectral_bandwidth)
    
    spectral_rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
    features['spectral_rolloff_mean'] = np.mean(spectral_rolloff)

    return features

def process_batch(file_list):
    all_features = []
    
    for file in file_list:
        print(f"Extrayendo caracteristicas de: {file}")
        file_features = extract_features(file)
        
        if file_features is not None:
            # Guardamos el nombre del archivo como identificador
            file_features['file_name'] = os.path.basename(file)
            all_features.append(file_features)
            
    # Convertir a un DataFrame de Pandas para facilitar el analisis tabular
    df = pd.DataFrame(all_features)
    
    # Reordenar columnas para que el nombre del archivo sea la primera
    cols = ['file_name'] + [c for c in df.columns if c != 'file_name']
    df = df[cols]
    
    return df

# --- Bloque de Ejecucion ---
if __name__ == "__main__":
    # Aqui debes incluir las rutas reales a tus archivos .wav, .mp3 o .flac
    archivos_de_prueba = [
        'ruta/a/tu/audio_real_1.wav', 
        'ruta/a/tu/audio_fake_1.wav'
    ] 
    
    # Si quieres procesar una carpeta entera, puedes usar librerias como glob:
    # import glob
    # archivos_de_prueba = glob.glob('ruta/a/tu/carpeta/*.wav')[:10] # Solo los primeros 10
    
    print("Iniciando procesamiento...")
    df_resultados = process_batch(archivos_de_prueba)
    
    if not df_resultados.empty:
        print("\nExtraccion completada con exito. Muestra de los datos:")
        print(df_resultados.head())
        
        # Opcional: Guardar los resultados para no tener que recalcularlos
        # df_resultados.to_csv('caracteristicas_audios.csv', index=False)

Iniciando procesamiento...
Extrayendo caracteristicas de: ruta/a/tu/audio_real_1.wav
Error al procesar ruta/a/tu/audio_real_1.wav: [Errno 2] No such file or directory: 'ruta/a/tu/audio_real_1.wav'
Extrayendo caracteristicas de: ruta/a/tu/audio_fake_1.wav
Error al procesar ruta/a/tu/audio_fake_1.wav: [Errno 2] No such file or directory: 'ruta/a/tu/audio_fake_1.wav'


C:\Users\Lechu\AppData\Local\Temp\ipykernel_9304\2183497171.py:10: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(file_path, sr=None)
c:\Users\Lechu\Desktop\Reto Telefonica\Reto_Inteligencia_Artificial_Liliana_Daniele_Alexis\.venv\Lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


KeyError: "None of [Index(['file_name'], dtype='str')] are in the [columns]"

In [ ]:
import glob

# Obtener los archivos de audio del dataset
ruta_audios = '../data/LA/ASVspoof2019_LA_train/flac/*.flac'
archivos_de_prueba = glob.glob(ruta_audios)

print(f"Se encontraron {len(archivos_de_prueba)} archivos de audio")
print("Iniciando procesamiento...")

# Procesar los archivos
df_resultados = process_batch(archivos_de_prueba[:100])  # Procesar los primeros 100 para eficiencia

if not df_resultados.empty:
    print("\nExtraccion completada con exito. Muestra de los datos:")
    print(df_resultados.head())
    
    # Guardar los resultados en CSV
    output_path = '../data/LA/caracteristicas_audios.csv'
    df_resultados.to_csv(output_path, index=False)
    print(f"\nResultados guardados en: {output_path}")
else:
    print("No se pudieron procesar los archivos.")

Se encontraron 25380 archivos de audio
Iniciando procesamiento...
Extrayendo caracteristicas de: ../data/LA/ASVspoof2019_LA_train/flac\LA_T_1000137.flac
Extrayendo caracteristicas de: ../data/LA/ASVspoof2019_LA_train/flac\LA_T_1000406.flac
Extrayendo caracteristicas de: ../data/LA/ASVspoof2019_LA_train/flac\LA_T_1000648.flac
Extrayendo caracteristicas de: ../data/LA/ASVspoof2019_LA_train/flac\LA_T_1000824.flac
Extrayendo caracteristicas de: ../data/LA/ASVspoof2019_LA_train/flac\LA_T_1001074.flac
Extrayendo caracteristicas de: ../data/LA/ASVspoof2019_LA_train/flac\LA_T_1001114.flac
Extrayendo caracteristicas de: ../data/LA/ASVspoof2019_LA_train/flac\LA_T_1001169.flac
Extrayendo caracteristicas de: ../data/LA/ASVspoof2019_LA_train/flac\LA_T_1001718.flac
Extrayendo caracteristicas de: ../data/LA/ASVspoof2019_LA_train/flac\LA_T_1001871.flac
Extrayendo caracteristicas de: ../data/LA/ASVspoof2019_LA_train/flac\LA_T_1002656.flac
Extrayendo caracteristicas de: ../data/LA/ASVspoof2019_LA_train/